# Моделирование. Проведение экспериментов. Подготовка пайплайнов обработки данных и построения модели.

# Импорты и настройки

In [41]:
# Стандартная библиотека
import os
import sys
import warnings
import joblib

# Работа с данными
import numpy as np
import pandas as pd

# ML
import lightgbm as lgb
import mlflow

# Настройки
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:,.0f}".format

# Пути проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.config import TRAIN_PATH, DATA_DIR, RAW_DIR

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Проверка путей
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", os.path.exists(TRAIN_PATH))
print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("RAW_DIR exists:", os.path.exists(RAW_DIR))
print("MODELS_DIR exists:", os.path.exists(MODELS_DIR))

PROJECT_ROOT: /home/mle_projects/bank-product-recs
TRAIN_PATH exists: True
DATA_DIR exists: True
RAW_DIR exists: True
MODELS_DIR exists: True


# Загрузка данных

In [7]:
df = pd.read_csv(TRAIN_PATH, low_memory=False)
print(df.shape)
df.head()

(13647309, 48)


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,ult_fec_cli_1t,indrel_1mes,tiprel_1mes,indresi,indext,conyuemp,canal_entrada,indfall,tipodom,cod_prov,nomprov,ind_actividad_cliente,renta,segmento,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,ind_deme_fin_ult1,ind_dela_fin_ult1,ind_ecue_fin_ult1,ind_fond_fin_ult1,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0,6,1,NaN,1.0,A,S,N,NaN,KHL,N,1,29,MALAGA,1,"87,218",02 - PARTICULARES,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0,35,1,NaN,1,I,S,S,NaN,KHE,N,1,13,CIUDAD REAL,0,"35,549",03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0,35,1,NaN,1,I,S,N,NaN,KHE,N,1,13,CIUDAD REAL,0,"122,179",03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0,35,1,NaN,1,I,S,N,NaN,KHD,N,1,50,ZARAGOZA,0,"119,776",03 - UNIVERSITARIO,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0,35,1,NaN,1,A,S,N,NaN,KHE,N,1,50,ZARAGOZA,1,NaN,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# Базовая предобработка

In [ ]:
# Даты
date_cols = ["fecha_dato", "fecha_alta", "ult_fec_cli_1t"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")


# Продуктовые признаки (таргет/фичи)
product_cols = [c for c in df.columns if c.endswith("_ult1")]

for col in product_cols:
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .fillna(0)
        .clip(0, 1)   # защита от мусора
        .astype(np.uint8)
    )


# Числовые признаки
num_cols = [
    "age",
    "antiguedad",
    "renta",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente"
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# Дополнительная очистка числовых
# возраст
if "age" in df.columns:
    df["age"] = df["age"].clip(18, 100)

# стаж
if "antiguedad" in df.columns:
    df["antiguedad"] = df["antiguedad"].clip(0, 500)

# доход (очень важный признак)
if "renta" in df.columns:
    df["renta"] = df["renta"].clip(0, df["renta"].quantile(0.99))
    df["renta_log"] = np.log1p(df["renta"])


# Категориальные признаки
cat_cols = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")


# Сортировка по времени
df = df.sort_values(["ncodpers", "fecha_dato"]).reset_index(drop=True)

# Формирование таргета

In [ ]:
# сортировка по клиенту и времени
df = df.sort_values(["ncodpers", "fecha_dato"]).copy()

# предыдущий продуктовый портфель клиента
prev_products = df.groupby("ncodpers")[product_cols].shift(1)

# признаки предыдущего состояния
prev_product_cols = [f"prev_{col}" for col in product_cols]
prev_products_for_features = prev_products.fillna(0).astype(np.uint8).copy()
prev_products_for_features.columns = prev_product_cols

# таргет: только новые продукты
target_df = (df[product_cols] - prev_products.fillna(0)).clip(lower=0).astype(np.uint8)
target_cols = [f"target_{col}" for col in product_cols]
target_df.columns = target_cols

# объединяем
df = pd.concat([df, prev_products_for_features, target_df], axis=1)

# удаляем первое наблюдение клиента, у которого нет истории
df["row_num_per_client"] = df.groupby("ncodpers").cumcount()
df = df[df["row_num_per_client"] > 0].copy()

print(df.shape)
print("Количество target-признаков:", len(target_cols))
print("Количество prev-признаков:", len(prev_product_cols))
print("NaN в таргете:", df[target_cols].isna().sum().sum())

(12690664, 73)


In [ ]:
# распределение таргета
df["new_products"] = df[target_cols].sum(axis=1)

print(df["new_products"].describe())
print(df["new_products"].value_counts(normalize=True).sort_index().head(10))

In [ ]:
# самые частые новые продукты
display(df[target_cols].sum().sort_values(ascending=False).head(10))

# Feature engineering

In [10]:
df["num_products"] = df[product_cols].sum(axis=1)
df["tenure_days"] = (df["fecha_dato"] - df["fecha_alta"]).dt.days
df["month"] = df["fecha_dato"].dt.month

#  Выбор признаков

In [ ]:
# Числовые
num_features = [
    "age",
    "antiguedad",
    "renta",
    "renta_log",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente",
]

# Категориальные
cat_features = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento",
]

num_features = [c for c in num_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]

feature_cols = num_features + cat_features + prev_product_cols
print("Количество признаков:", len(feature_cols))

## Обработка категорий

### Для LightGBM можно закодировать категории через category.

In [14]:
for col in cat_features:
    df[col] = df[col].fillna("UNKNOWN").astype(str).astype("category")

### Числовые:

In [15]:
for col in num_features:
    df[col] = df[col].fillna(df[col].median())

## Time-based split

In [16]:
train_df = df[df["fecha_dato"] < "2016-05-28"].copy()
valid_df = df[df["fecha_dato"] == "2016-05-28"].copy()

print(train_df.shape, valid_df.shape)

(11763904, 76) (926760, 76)


# Обучене модели 

## Baseline Top Popular

### Считаем самые популярные новые продукты на train:

In [52]:
popular_targets = train_df[target_cols].sum().sort_values(ascending=False)
top_popular_products = [c.replace("target_", "") for c in popular_targets.index.tolist()]
top_popular_products[:5]

['ind_recibo_ult1',
 'ind_nom_pens_ult1',
 'ind_nomina_ult1',
 'ind_cco_fin_ult1',
 'ind_tjcr_fin_ult1']

## Метрики

In [47]:
def precision_at_k(actual, predicted, k=5):
    predicted = predicted[:k]
    return len(set(actual) & set(predicted)) / k if k > 0 else 0.0


def recall_at_k(actual, predicted, k=5):
    predicted = predicted[:k]
    return len(set(actual) & set(predicted)) / len(actual) if len(actual) > 0 else 0.0


def apk(actual, predicted, k=5):
    predicted = predicted[:k]
    score = 0.0
    hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            hits += 1.0
            score += hits / (i + 1.0)

    return score / min(len(actual), k) if len(actual) > 0 else 0.0


def mean_metrics(actual_list, predicted_list, k=5):
    precisions = [precision_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    recalls = [recall_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    maps = [apk(a, p, k) for a, p in zip(actual_list, predicted_list)]

    return {
        f"precision_at_{k}": float(np.mean(precisions)),
        f"recall_at_{k}": float(np.mean(recalls)),
        f"map_at_{k}": float(np.mean(maps)),
    }

## Подготовка фактических ответов для valid

In [29]:
def extract_actual_products(row, target_cols):
    return [c.replace("target_", "") for c in target_cols if row[c] == 1]

valid_actual = valid_df[target_cols].apply(
    lambda row: extract_actual_products(row, target_cols),
    axis=1
).tolist()

In [30]:
# Baseline predictions:
K = 5

popular_targets = train_df[target_cols].sum().sort_values(ascending=False)
top_popular_products = [c.replace("target_", "") for c in popular_targets.index.tolist()]

valid_pred_top_pop = [top_popular_products[:K] for _ in range(len(valid_df))]

baseline_metrics = mean_metrics(valid_actual, valid_pred_top_pop, k=K)

print("Baseline metrics:", baseline_metrics)

Baseline metrics: {'precision_at_5': 0.00632094609176054, 'recall_at_5': 0.024207364006502932, 'map_at_5': 0.015614864629941346}


## Обучение LightGBM

### по одной модели на каждый продукт.

In [ ]:
models = {}
valid_scores = pd.DataFrame(index=valid_df.index)

X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

for col in cat_features:
    X_train[col] = X_train[col].astype("category")
    X_valid[col] = X_valid[col].astype("category")

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": 42,
    "is_unbalance": True,
}

MIN_POSITIVES = 300

for i, target_col in enumerate(target_cols, start=1):
    y_train = train_df[target_col].astype(int)
    y_valid = valid_df[target_col].astype(int)

    n_pos_train = int(y_train.sum())
    n_pos_valid = int(y_valid.sum())

    print(
        f"[{i}/{len(target_cols)}] {target_col} | "
        f"train positives={n_pos_train}, valid positives={n_pos_valid}"
    )

    # слишком редкий таргет — модель не обучаем
    if n_pos_train < MIN_POSITIVES:
        print(f"  -> skip: too few positives in train (< {MIN_POSITIVES})")
        valid_scores[target_col] = 0.0
        continue

    # защита: в train должны быть оба класса
    if y_train.nunique() < 2:
        print("  -> skip: only one class in train")
        valid_scores[target_col] = 0.0
        continue

    train_dataset = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_features,
        free_raw_data=False
    )

    valid_dataset = lgb.Dataset(
        X_valid,
        label=y_valid,
        categorical_feature=cat_features,
        free_raw_data=False
    )

    model = lgb.train(
        params,
        train_dataset,
        valid_sets=[valid_dataset],
        num_boost_round=300,
        callbacks=[
            lgb.early_stopping(30),
            lgb.log_evaluation(0)
        ]
    )

    models[target_col] = model
    valid_scores[target_col] = model.predict(
        X_valid,
        num_iteration=model.best_iteration
    )

print(f"\nОбучено моделей: {len(models)} из {len(target_cols)}")
print("Форма valid_scores:", valid_scores.shape)

[1/24] target_ind_ahor_fin_ult1 | train positives=1, valid positives=1
  -> skip: too few positives in train (< 300)
[2/24] target_ind_aval_fin_ult1 | train positives=4, valid positives=0
  -> skip: too few positives in train (< 300)
[3/24] target_ind_cco_fin_ult1 | train positives=66119, valid positives=3878
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[300]	valid_0's binary_logloss: 0.281255
[4/24] target_ind_cder_fin_ult1 | train positives=131, valid positives=5
  -> skip: too few positives in train (< 300)
[5/24] target_ind_cno_fin_ult1 | train positives=34840, valid positives=2347
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[2]	valid_0's binary_logloss: 0.0805573
[6/24] target_ind_ctju_fin_ult1 | train positives=450, valid positives=40
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[2]	valid_0's binary_logloss: 0.06279

## Формирование рекомендаций модели

### Важно не рекомендовать уже имеющиеся у клиента продукты.

In [53]:
def get_model_recommendations(valid_df, score_df, product_cols, top_k=5):
    recommendations = []

    for idx in valid_df.index:
        # текущие продукты клиента
        current_products = set([p for p in product_cols if valid_df.loc[idx, p] == 1])

        # вероятности
        scores = {
            target_col.replace("target_", ""): score_df.loc[idx, target_col]
            for target_col in score_df.columns
        }

        # исключаем уже имеющиеся продукты
        filtered_scores = {p: s for p, s in scores.items() if p not in current_products}

        # сортировка
        ranked = sorted(filtered_scores.items(), key=lambda x: x[1], reverse=True)
        recs = [p for p, _ in ranked[:top_k]]

        recommendations.append(recs)

    return recommendations

In [54]:
valid_pred_lgb = get_model_recommendations(
    valid_df,
    valid_scores,
    product_cols,
    top_k=K
)

lgb_metrics = mean_metrics(valid_actual, valid_pred_lgb, k=K)

print("LightGBM metrics:", lgb_metrics)

LightGBM metrics: {'precision_at_5': 0.0, 'recall_at_5': 0.0, 'map_at_5': 0.0}


## Логирование в MLflow

In [49]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("bank_product_recommendation")

with mlflow.start_run(run_name="lightgbm_one_vs_rest_v1"):

    # параметры
    mlflow.log_param("model_type", "lightgbm_one_vs_rest")
    mlflow.log_param("top_k", K)
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("num_products", len(product_cols))

    # baseline метрики
    for metric_name, metric_value in baseline_metrics.items():
        mlflow.log_metric(f"baseline_{metric_name}", float(metric_value))

    # модельные метрики
    for metric_name, metric_value in lgb_metrics.items():
        mlflow.log_metric(metric_name, float(metric_value))

print("MLflow logging completed")

🏃 View run lightgbm_one_vs_rest_v1 at: http://127.0.0.1:5000/#/experiments/1/runs/828f8e067e794825899664f27009a501
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
MLflow logging completed


## Сохранение модели

In [50]:
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# pkl
joblib.dump(models, os.path.join(MODELS_DIR, "lgbm_one_vs_rest.pkl"))

# bin
for target_col, model in models.items():
    model.save_model(os.path.join(MODELS_DIR, f"{target_col}.bin"))

print("Модели сохранены")

Модели сохранены


In [51]:
print("TRAIN target sum:", train_df[target_cols].sum().sum())
print("VALID target sum:", valid_df[target_cols].sum().sum())

valid_actual = valid_df[target_cols].apply(
    lambda row: [c.replace("target_", "") for c in target_cols if row[c] == 1],
    axis=1
).tolist()

num_non_empty_actual = sum(len(x) > 0 for x in valid_actual)
print("Non-empty actual in valid:", num_non_empty_actual)
print("Share:", round(num_non_empty_actual / len(valid_actual), 4))

print("\nTop valid targets:")
display(valid_df[target_cols].sum().sort_values(ascending=False).head(10))

print("\nScore stats:")
display(valid_scores.describe())

print("\nExamples:")
for i in range(5):
    print("ACTUAL:", valid_actual[i])
    print("PRED  :", valid_pred_lgb[i])
    print("-" * 40)

TRAIN target sum: 527422.0
VALID target sum: 35888.0
Non-empty actual in valid: 27916
Share: 0.0301

Top valid targets:


target_ind_recibo_ult1     10,163
target_ind_nom_pens_ult1    5,513
target_ind_nomina_ult1      5,488
target_ind_tjcr_fin_ult1    4,248
target_ind_cco_fin_ult1     3,878
target_ind_ecue_fin_ult1    2,709
target_ind_cno_fin_ult1     2,347
target_ind_ctma_fin_ult1      531
target_ind_reca_fin_ult1      279
target_ind_ctop_fin_ult1      226
dtype: float64


Score stats:


,target_ind_ahor_fin_ult1,target_ind_aval_fin_ult1,target_ind_cco_fin_ult1,target_ind_cder_fin_ult1,target_ind_cno_fin_ult1,target_ind_ctju_fin_ult1,target_ind_ctma_fin_ult1,target_ind_ctop_fin_ult1,target_ind_ctpp_fin_ult1,target_ind_deco_fin_ult1,target_ind_deme_fin_ult1,target_ind_dela_fin_ult1,target_ind_ecue_fin_ult1,target_ind_fond_fin_ult1,target_ind_hip_fin_ult1,target_ind_plan_fin_ult1,target_ind_pres_fin_ult1,target_ind_reca_fin_ult1,target_ind_tjcr_fin_ult1,target_ind_valo_fin_ult1,target_ind_viv_fin_ult1,target_ind_nomina_ult1,target_ind_nom_pens_ult1,target_ind_recibo_ult1
count,"926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760"
mean,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
std,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
min,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
25%,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
50%,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
75%,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
max,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1



Examples:
ACTUAL: ['ind_tjcr_fin_ult1']
PRED  : ['ind_dela_fin_ult1', 'ind_ctma_fin_ult1', 'ind_reca_fin_ult1', 'ind_deco_fin_ult1', 'ind_ctop_fin_ult1']
----------------------------------------
ACTUAL: []
PRED  : ['ind_dela_fin_ult1', 'ind_ctma_fin_ult1', 'ind_valo_fin_ult1', 'ind_reca_fin_ult1', 'ind_deco_fin_ult1']
----------------------------------------
ACTUAL: []
PRED  : ['ind_ctma_fin_ult1', 'ind_deco_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_ctop_fin_ult1', 'ind_fond_fin_ult1']
----------------------------------------
ACTUAL: []
PRED  : ['ind_dela_fin_ult1', 'ind_ctma_fin_ult1', 'ind_deco_fin_ult1', 'ind_reca_fin_ult1', 'ind_ctpp_fin_ult1']
----------------------------------------
ACTUAL: []
PRED  : ['ind_dela_fin_ult1', 'ind_ctma_fin_ult1', 'ind_deco_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_ctop_fin_ult1']
----------------------------------------
